### 预先注册的动态工具
#### 根据 state 选择工具

In [1]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from collections.abc import Callable
from langchain.tools import tool
from typing import TypedDict, Annotated
from langchain.messages import AnyMessage
from langchain.chat_models import init_chat_model
from agent_lab.model_hub import LLM_FAST
import operator

@tool
def public_search(query: str) -> str:
    """Search the web for the given query and return a summary of the results. All agent requests can use this tool."""
    return f"Search results for '{query}'"

@tool
def private_search(query: str) -> str:
    """Search the web for the given query and return a summary of the results. This tool can be used when the agent is authorized."""
    return f"Private search results for '{query}'"

@tool
def advanced_search(query: str) -> str:
    """Search the web for the given query and return a summary of the results. This tool is used for deep searching and can only be utilized when the agent is authorized and the number of messages exceeds 5."""
    return f"Advanced search results for '{query}'"

@wrap_model_call
def state_based_tools(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Filter tools based on conversation State."""
    # Read from State: check if user has authenticated
    state = request.state
    is_authenticated = state.get("authenticated", False)
    message_count = len(state["messages"])

    # Only enable sensitive tools after authentication
    if not is_authenticated:
        tools = [t for t in request.tools if t.name.startswith("public_")] # type: ignore
        request = request.override(tools=tools)
    elif message_count < 6:
        # Limit tools early in conversation
        tools = [t for t in request.tools if t.name != "advanced_search"] # type: ignore
        request = request.override(tools=tools)

    return handler(request)

class AgentState(TypedDict):
    authenticated: bool
    messages: Annotated[list[AnyMessage], operator.add]

agent = create_agent(
    model=init_chat_model(**LLM_FAST),
    tools=[public_search, private_search, advanced_search],
    middleware=[state_based_tools],
    state_schema=AgentState, # type: ignore
)

In [2]:
from langchain.messages import HumanMessage

response1 = agent.invoke({
    'messages': [
        HumanMessage('How many search tools do you have available? Please list them.')
    ]
})

In [3]:
response1['messages'][-1].pretty_print()

================================== Ai Message ==================================

Based on the information provided to me, I have access to **one search tool**. Here are the details:

### Available Search Tool:

1. **`public_search`**
   - **Description:** Search the web for a given query and return a summary of the results.
   - **Parameters:**
     - `query` (string, required): The search query to look up.

That's the only search tool I have available in my current toolkit. Let me know if you'd like me to search for anything!


In [4]:
response2 = agent.invoke(
    {
        'messages': [
            HumanMessage('How many search tools do you have available? Please list them.')
        ],
        'authenticated': True,  # Simulate user authentication
    } # type: ignore
)

In [5]:
response2['messages'][-1].pretty_print()

================================== Ai Message ==================================

Let me check what tools I have access to.

Based on my available tools, I have **2 search tools** available:

1. **`public_search`** - Use this to search the web for a given query and return a summary of the results. The note says "All agent requests can use this tool."

2. **`private_search`** - Use this to search the web for a given query and return a summary of the results. This tool can be used when the agent is authorized.

Both tools take a single parameter called `query` (a string). The main difference is that `public_search` is available for all requests, while `private_search` requires authorization.


In [6]:
from langchain.messages import SystemMessage, AIMessage

response3 = agent.invoke(
    {
        'messages': [
            SystemMessage('You are an assistant that helps users search the web.'),
            HumanMessage('Hello.'),
            AIMessage('Hi there! How can I assist you today?'),
            HumanMessage('I want to ask you some questions.'),
            AIMessage('Sure! What would you like to know?'),
            HumanMessage('How many search tools do you have available? Please list them.'),
        ],
        'authenticated': True,  # Simulate user authentication
    } # type: ignore
)

In [7]:
response3['messages'][-1].pretty_print()

================================== Ai Message ==================================

I have **3 search tools** available:

1. **public_search** - Searches the web for the given query and returns a summary of the results. This is available for all requests.

2. **private_search** - Also searches the web and returns a summary. This can be used when I'm authorized.

3. **advanced_search** - Searches the web and returns a summary. This is used for deep searching and can only be utilized when I'm authorized and the number of messages in the conversation exceeds 5.

Would you like me to search for something?


#### 根据 Store 选择工具

In [8]:
from dataclasses import dataclass
from langgraph.store.memory import InMemoryStore

@dataclass
class Context:
    user_id: str

@wrap_model_call
def store_based_tools(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Filter tools based on Store preferences."""
    user_id = request.runtime.context.user_id # type: ignore

    # Read from Store: get user's enabled features
    feature_flags = request.runtime.store.get(("features",), user_id) # type: ignore

    if feature_flags:
        enabled_features = feature_flags.value.get("enabled_tools", [])
        # Only include tools that are enabled for this user
        tools = [t for t in request.tools if t.name in enabled_features] # type: ignore
        request = request.override(tools=tools)

    return handler(request)

store = InMemoryStore()
store.put(("features",), "user123", {"enabled_tools": ["public_search", "advanced_search"]})
store.put(("features",), "user456", {"enabled_tools": ["public_search", "private_search", "advanced_search"]})

agent = create_agent(
    model=init_chat_model(**LLM_FAST),
    tools=[public_search, private_search, advanced_search],
    middleware=[store_based_tools], # type: ignore
    context_schema=Context,
    store=store,
)

In [9]:
response4 = agent.invoke(
    {'messages': [HumanMessage('How many search tools do you have available? Please list them.')]},
    context=Context(user_id='user123'),
)

In [10]:
response4['messages'][-1].pretty_print()

================================== Ai Message ==================================

I have **2 search tools** available:

1. **`public_search`** - Search the web for a given query and return a summary of the results. This tool is available for all agent requests to use.

2. **`advanced_search`** - Search the web for a given query and return a summary of the results. This tool is used for deep searching and can only be utilized when the agent is authorized and the number of messages exceeds 5.


In [11]:
response5 = agent.invoke(
    {'messages': [HumanMessage('How many search tools do you have available? Please list them.')]},
    context=Context(user_id='user456'),
)

In [12]:
response5['messages'][-1].pretty_print()

================================== Ai Message ==================================

Let me check the available tools I have access to.

Based on the function definitions provided to me, I have **3 search tools** available:

1. **`public_search`** - Searches the web for the given query and returns a summary of the results. All agent requests can use this tool.

2. **`private_search`** - Searches the web for the given query and returns a summary of the results. This tool can be used when the agent is authorized.

3. **`advanced_search`** - Searches the web for the given query and returns a summary of the results. This tool is used for deep searching and can only be utilized when the agent is authorized and the number of messages exceeds 5.

All three tools accept a single parameter: `query` (a string), and they all return a summary of search results from the web. The main differences are the **access levels**:
- **`public_search`**: Available to everyone, no restrictions.
- **`private_sear

#### 根据 Context 选择工具

In [13]:
@dataclass
class Context:
    user_role: str

@wrap_model_call
def context_based_tools(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Filter tools based on Runtime Context permissions."""
    # Read from Runtime Context: get user role
    if request.runtime is None or request.runtime.context is None:
        # If no context provided, default to visitor (most restrictive)
        user_role = "visitor"
    else:
        user_role = request.runtime.context.user_role

    if user_role == "pro":
        # Admins get all tools
        pass
    elif user_role == "user":
        tools = [t for t in request.tools if t.name != "advanced_search"] # type: ignore
        request = request.override(tools=tools)
    else:
        tools = [t for t in request.tools if t.name.startswith("public_")] # type: ignore
        request = request.override(tools=tools)

    return handler(request)

agent = create_agent(
    model=init_chat_model(**LLM_FAST),
    tools=[public_search, private_search, advanced_search],
    middleware=[context_based_tools], # type: ignore
    context_schema=Context,
)

In [14]:
response6 = agent.invoke(
    {'messages': [HumanMessage('How many search tools do you have available? Please list them.')]},
    context=Context(user_role='pro'),
)

In [15]:
response6['messages'][-1].pretty_print()

================================== Ai Message ==================================

I have **3 search tools** available to me. Here they are:

1. **`public_search`** - Searches the web for a given query and returns a summary of the results. Available for all agent requests.

2. **`private_search`** - Searches the web for a given query and returns a summary of the results. Can be used when the agent is authorized.

3. **`advanced_search`** - Searches the web for a given query and returns a summary of the results. Used for deep searching and can only be utilized when the agent is authorized **and** the number of messages exceeds 5.


In [16]:
response7 = agent.invoke(
    {'messages': [HumanMessage('How many search tools do you have available? Please list them.')]},
    context=Context(user_role='user'),
)

In [17]:
response7['messages'][-1].pretty_print()

================================== Ai Message ==================================

I have **two search tools** available:

1. **`public_search`** - Searches the web for a given query and returns a summary of the results. This tool is available for all agent requests to use.

2. **`private_search`** - Also searches the web for a given query and returns a summary of the results. This tool can be used when the agent is authorized (i.e., it's intended for authorized/privileged use cases).


In [18]:
response8 = agent.invoke(
    {'messages': [HumanMessage('How many search tools do you have available? Please list them.')]},
    context=Context(user_role='viewer'),
)

In [19]:
response8['messages'][-1].pretty_print()

================================== Ai Message ==================================

I have access to **1 search tool**, which is:

1. **`public_search`** – This tool allows me to search the web for a given query and return a summary of the results.

It accepts a single parameter:
- **`query`** (string) – The search term you want to look up.

That's the only tool currently available to me. Let me know if you'd like me to search for something!


### 运行时注册工具

In [20]:
from langchain.agents.middleware import AgentMiddleware, ToolCallRequest

@tool
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    assert city != ''
    return f"It's always sunny in {city}."

# A tool that will be added dynamically at runtime
@tool
def calculate_tip(bill_amount: float, tip_percentage: float = 20.0) -> str:
    """Calculate the tip amount for a bill."""
    tip = bill_amount * (tip_percentage / 100)
    return f"Tip: ${tip:.2f}, Total: ${bill_amount + tip:.2f}"

class DynamicToolMiddleware(AgentMiddleware):
    """Middleware that registers and handles dynamic tools."""

    def wrap_model_call(self, request: ModelRequest, handler):
        # Add dynamic tool to the request
        # This could be loaded from an MCP server, database, etc.
        updated = request.override(tools=[*request.tools, calculate_tip])
        return handler(updated)

    def wrap_tool_call(self, request: ToolCallRequest, handler):
        # Handle execution of the dynamic tool
        if request.tool_call["name"] == "calculate_tip":
            return handler(request.override(tool=calculate_tip))
        return handler(request)

agent = create_agent(
    model=init_chat_model(**LLM_FAST),
    tools=[get_weather],  # Only static tools registered here
    middleware=[DynamicToolMiddleware()],
)

# The agent can now use both get_weather AND calculate_tip
result = agent.invoke({
    "messages": [{"role": "user", "content": "Calculate a 20% tip on $85"}]
})

In [21]:
from pprint import pprint

pprint(result)

{'messages': [HumanMessage(content='Calculate a 20% tip on $85', additional_kwargs={}, response_metadata={}, id='6736af53-5d73-410f-8401-5c79216e3696'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 352, 'total_tokens': 415, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 352}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_058df29938_prod0820_fp8_kvcache_20260402', 'id': '44a8fa9a-a420-40a2-b95e-51665d715bc5', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dd218-9f0d-7b23-a60d-b7bbe18c4d80-0', tool_calls=[{'name': 'calculate_tip', 'args': {'bill_amount': 85, 'tip_percentage': 20}, 'id': 'call_00_z9jCWAm7z9lNOFuyb3myByxU', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 352, 'outpu